In [46]:
import pandas as pd
import matplotlib.pyplot as plt

In [47]:
data = pd.read_csv("./data/training_data_lowercase.csv",sep='\t', names=['label', 'title'])
print(data.shape)
data.fillna("",inplace=True)
print(data.head())


(34152, 2)
   label                                              title
0      0  donald trump sends out embarrassing new year‚s...
1      0  drunk bragging trump staffer started russian c...
2      0  sheriff david clarke becomes an internet joke ...
3      0  trump is so obsessed he even has obama‚s name ...
4      0  pope francis just called out donald trump duri...


as part of pre proc we are able to see color codes in csv which are incorrectly interpretted by vs code
they are not color code but simply corresponds to episode num

we also see video/picture are there in some data points, while these are just metadata it could change the meaning of the sentence once we remove punctuations

we also see that this metadata in enclosed in () in Training sample while its enclosed in [] in testing data, so we might need different pre processing

In [48]:
from preProc import normalize_text

data["clean_text"] = data["title"].apply(normalize_text)
print(data.head)

<bound method NDFrame.head of        label                                              title  \
0          0  donald trump sends out embarrassing new year‚s...   
1          0  drunk bragging trump staffer started russian c...   
2          0  sheriff david clarke becomes an internet joke ...   
3          0  trump is so obsessed he even has obama‚s name ...   
4          0  pope francis just called out donald trump duri...   
...      ...                                                ...   
34147      1  tears in rain as thais gather for late king's ...   
34148      1  pyongyang university needs non-u.s. teachers a...   
34149      1  philippine president duterte to visit japan ah...   
34150      1  japan's abe may have won election\tbut many do...   
34151      1  demoralized and divided: inside catalonia's po...   

                                              clean_text  
0      donald trump sends out embarrassing new year s...  
1      drunk bragging trump staffer started rus

In [49]:
from preProc import remove_stopwords


data["no_stopwords"] = data["clean_text"].apply(remove_stopwords)
print(data.head)



<bound method NDFrame.head of        label                                              title  \
0          0  donald trump sends out embarrassing new year‚s...   
1          0  drunk bragging trump staffer started russian c...   
2          0  sheriff david clarke becomes an internet joke ...   
3          0  trump is so obsessed he even has obama‚s name ...   
4          0  pope francis just called out donald trump duri...   
...      ...                                                ...   
34147      1  tears in rain as thais gather for late king's ...   
34148      1  pyongyang university needs non-u.s. teachers a...   
34149      1  philippine president duterte to visit japan ah...   
34150      1  japan's abe may have won election\tbut many do...   
34151      1  demoralized and divided: inside catalonia's po...   

                                              clean_text  \
0      donald trump sends out embarrassing new year s...   
1      drunk bragging trump staffer started r

In [50]:
from preProc import tokens_lemm, tokens_stemm

data["stemmed"] = data["clean_text"].apply(tokens_stemm)
data["lemmed"] = data["clean_text"].apply(tokens_lemm)

print(data.head)


<bound method NDFrame.head of        label                                              title  \
0          0  donald trump sends out embarrassing new year‚s...   
1          0  drunk bragging trump staffer started russian c...   
2          0  sheriff david clarke becomes an internet joke ...   
3          0  trump is so obsessed he even has obama‚s name ...   
4          0  pope francis just called out donald trump duri...   
...      ...                                                ...   
34147      1  tears in rain as thais gather for late king's ...   
34148      1  pyongyang university needs non-u.s. teachers a...   
34149      1  philippine president duterte to visit japan ah...   
34150      1  japan's abe may have won election\tbut many do...   
34151      1  demoralized and divided: inside catalonia's po...   

                                              clean_text  \
0      donald trump sends out embarrassing new year s...   
1      drunk bragging trump staffer started r

In [51]:
from sklearn.model_selection import train_test_split
X=data.drop(columns=['label'])
y=data['label']

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print(f"Training set size: {X_train.shape}")
print(f"Testing set size: {X_val.shape}")
print(f"Training output size: {y_train.shape}")
print(f"Testing output size: {y_val.shape}")

Training set size: (27321, 5)
Testing set size: (6831, 5)
Training output size: (27321,)
Testing output size: (6831,)


* Use separate vectorizers, in model 1 we used the same vectorizer twice with fit_transform which verwrites vocabulary → inconsistent features
* Add raw cleaned text

BOW

In [52]:
from sklearn.feature_extraction.text import CountVectorizer
bow_vectorizer_lemm = CountVectorizer(max_features=5000)
X_train_BOW_L = bow_vectorizer_lemm.fit_transform(X_train["lemmed"])
X_val_BOW_L = bow_vectorizer_lemm.transform(X_val["lemmed"])

bow_vectorizer_stemm = CountVectorizer(max_features=5000)
X_train_BOW_S = bow_vectorizer_stemm.fit_transform(X_train["stemmed"])
X_val_BOW_S = bow_vectorizer_stemm.transform(X_val["stemmed"])

bow_vectorizer_raw = CountVectorizer(max_features=5000)
X_train_BOW_raw = bow_vectorizer_raw.fit_transform(X_train["clean_text"])
X_val_BOW_raw = bow_vectorizer_raw.transform(X_val["clean_text"])

TFIDF

In [53]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_lemm = TfidfVectorizer(
    max_features=15000,
    ngram_range=(1,2),   
    min_df=3,
    max_df=0.85
)
X_train_TF_L = tfidf_lemm.fit_transform(X_train["lemmed"])
X_val_TF_L = tfidf_lemm.transform(X_val["lemmed"])

tfidf_stemm = TfidfVectorizer(
    max_features=15000,
    ngram_range=(1,2),   
    min_df=3,
    max_df=0.85
)
X_train_TF_S = tfidf_stemm.fit_transform(X_train["stemmed"])
X_val_TF_S = tfidf_stemm.transform(X_val["stemmed"])

tfidf_raw = TfidfVectorizer(
    max_features=15000,
    ngram_range=(1,2),   
    min_df=3,
    max_df=0.85
)
X_train_TF_raw = tfidf_raw.fit_transform(X_train["clean_text"])
X_val_TF_raw = tfidf_raw.transform(X_val["clean_text"])

In [54]:
from scipy.sparse import hstack, csr_matrix

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier


from sklearn.metrics import accuracy_score, classification_report, confusion_matrix



experiments = {
    "BoW_stemm": (X_train_BOW_S, X_val_BOW_S),
    "BoW_lemm": (X_train_BOW_L, X_val_BOW_L),
    "BoW_row": (X_train_BOW_raw, X_val_BOW_raw),
    "TF-IDF_stemm": (X_train_TF_S, X_val_TF_S),
    "TF-IDF_lemm": (X_train_TF_L, X_val_TF_L),
    "TF-IDF_row": (X_train_TF_raw, X_val_TF_raw)
}

X_train_combined = hstack([X_train_TF_raw, X_train_TF_S])
X_val_combined = hstack([X_val_TF_raw, X_val_TF_S])

experiments["TF-IDF_combined"] = (X_train_combined, X_val_combined)

models = {
    "Naive Bayes": MultinomialNB(),
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Linear SVC": LinearSVC(),
}

results = []

for name, (Xtr, Xva) in experiments.items():
    for model_name, model in models.items():
        model.fit(Xtr, y_train)
        preds = model.predict(Xva)
        acc = accuracy_score(y_val, preds)
        results.append((name, model_name, acc))
        print(f"{name} : {model_name} : {acc:.4f}")
 

results_df = pd.DataFrame(results, columns=["Text Vectorization", "Model", "Validation Accuracy"]).sort_values(
    by="Validation Accuracy", ascending=False
)
print("\nValidation results:")
display(results_df)

best_model_name = results_df.iloc[0]["Model"]
print("Best model:", best_model_name)

# best_X_train, best_X_val = experiments[best_model_name]
# best_model = MultinomialNB()
# best_model.fit(best_X_train, y_train)
# best_preds = best_model.predict(best_X_val)

# print("\nClassification report for best model:")
# print(classification_report(y_val, best_preds))
# print("Confusion matrix:")
# print(confusion_matrix(y_val, best_preds))

BoW_stemm : Naive Bayes : 0.9340
BoW_stemm : Logistic Regression : 0.9428
BoW_stemm : Linear SVC : 0.9356
BoW_lemm : Naive Bayes : 0.9366
BoW_lemm : Logistic Regression : 0.9431
BoW_lemm : Linear SVC : 0.9363
BoW_row : Naive Bayes : 0.9409
BoW_row : Logistic Regression : 0.9477
BoW_row : Linear SVC : 0.9368
TF-IDF_stemm : Naive Bayes : 0.9368
TF-IDF_stemm : Logistic Regression : 0.9467
TF-IDF_stemm : Linear SVC : 0.9530
TF-IDF_lemm : Naive Bayes : 0.9385
TF-IDF_lemm : Logistic Regression : 0.9472
TF-IDF_lemm : Linear SVC : 0.9524
TF-IDF_row : Naive Bayes : 0.9376
TF-IDF_row : Logistic Regression : 0.9463
TF-IDF_row : Linear SVC : 0.9530
TF-IDF_combined : Naive Bayes : 0.9391
TF-IDF_combined : Logistic Regression : 0.9515
TF-IDF_combined : Linear SVC : 0.9537

Validation results:


,Text Vectorization,Model,Validation Accuracy
20,TF-IDF_combined,Linear SVC,0.953740
17,TF-IDF_row,Linear SVC,0.953008
11,TF-IDF_stemm,Linear SVC,0.953008
14,TF-IDF_lemm,Linear SVC,0.952423
19,TF-IDF_combined,Logistic Regression,0.951544
7,BoW_row,Logistic Regression,0.947738
13,TF-IDF_lemm,Logistic Regression,0.947153
10,TF-IDF_stemm,Logistic Regression,0.946714
16,TF-IDF_row,Logistic Regression,0.946274
4,BoW_lemm,Logistic Regression,0.943054


Best model: Linear SVC
